# 3. From motif counts to a linear expression model (Fig. 1h, one biosample)

Fig. 1h asks how much of the expression signal the motif vocabulary carries when the sequence is reduced to **motif counts**. This notebook fits the paper's ridge model on the shipped K562 count matrices (`data/k562_demo/count_matrices/`) for de novo versus JASPAR motifs, peak-weighted versus unweighted counting, and promoter / enhancer / both feature sets, and compares the held-out correlations with the published K562 values. CPU, a few minutes, nothing to download.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/setup_work_dir.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
os.environ["ENHANCER_ROOT"], os.environ["SEQ_ROOT"] = ENHANCER_ROOT, SEQ_ROOT   # the scripts in code/ read their data root from here
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    if cmd.startswith("python "): cmd = f'"{sys.executable}" ' + cmd[7:]   # the interpreter this notebook runs on, not whatever comes first on PATH
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
# ---- CONFIGURATION ---------------------------------------------------------------------------------------------
BSSID = "BSS00762"            # K562. Any of the 116 biosamples works once its two input files are in INPUTS (Zenodo fig1_training_inputs_all116)
N_MODELS = 1                  # models to train in section 1 (one per training fold); the paper trained 10
EPOCHS = 100                  # maximum epochs per model (paper: 100, early stopping with patience 10 on the validation fold)
SEED = 42
RUN_TRAINING = True           # False: skip training (no GPU) and use the released K562 fold-0 model and predictions in data/k562_sample
VALIDATE_EXTRACTION = True    # notebook 02, 1.3: extract from the released K562 fold-0 weights and compare with the released PWMs (10 min on a GPU)
RUN_TOMTOM_DEMO = True        # notebook 02, 3.3: run TOMTOM for the K562 promoter set against JASPAR here (about 4 min) and compare with the shipped table
INPUTS     = f"{ENHANCER_ROOT}/fig1_training_inputs" if BSSID != "BSS00762" else os.path.join(REPO, "data", "k562_demo")   # K562 ships in the repository; the other 115 biosamples: Zenodo fig1_training_inputs_all116 unpacked under ENHANCER_ROOT
MOTIF_SETS = f"{ENHANCER_ROOT}/fig1_motif_sets"         # Zenodo fig1_motif_sets_all116.tar.gz unpacked under ENHANCER_ROOT (section 4)
DEMO   = os.path.join(REPO, "data", "k562_demo")        # shipped: motifs + TOMTOM tables of the ten K562 models, K562 count matrices, K562 expression
SAMPLE = os.path.join(REPO, "data", "k562_sample")      # shipped: released K562 fold-0 model and the held-out predictions of the ten released models
WORK   = f"{ENHANCER_ROOT}/notebook_runs/fig1/{BSSID}"; os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, f"{CODE}/cnncr_model"); sys.path.insert(0, f"{CODE}/motif_extraction")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, logomaker
from scipy.stats import spearmanr, pearsonr
from IPython.display import Image, display
from cnncr_io import read_windows_fasta, read_expression, one_hot      # code/cnncr_model/cnncr_io.py
import cnncr_model as cm                                                # code/cnncr_model/cnncr_model.py: the model, split and training loop of the paper
from extract_motifs_lib import extract_model, write_meme, read_meme     # code/motif_extraction/extract_motifs_lib.py
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device, "| work folder:", WORK)
PRO, PRO_F, ENH, ENH_F = "#8FB8C9", "#D6E7EE", "#D79581", "#F4E0D8"        # promoter / enhancer colours of the paper's figures
def show_pwms(pwms, titles, suptitle=""):
    """sequence logos (information content) of a few PWMs"""
    fig, axes = plt.subplots(1, len(pwms), figsize=(2.0*len(pwms), 1.3))
    for ax, p, t in zip(np.atleast_1d(axes), pwms, titles):
        logomaker.Logo(logomaker.transform_matrix(pd.DataFrame(p, columns=list("ACGT")), from_type="probability", to_type="information"), ax=ax)
        ax.set_title(t, fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(suptitle, fontsize=9); plt.tight_layout(); plt.show()
def spearman(a, b): return spearmanr(a, b)[0]
def same_pwms(a, b):
    """the same PWMs regardless of order (each PWM rounded to 6 decimals)"""
    key = lambda m: np.round(m, 6).tobytes(); ka, kb = set(map(key, a)), set(map(key, b))
    return f"{len(ka & kb)} of {len(kb)} released PWMs recovered" + (" (identical set)" if len(a) == len(b) and ka == kb else "")

## 1. From motif counts to a linear expression model (Fig. 1h, one biosample)

Fig. 1h asks how much of the expression signal the motif vocabulary carries when the sequence is reduced to **motif counts**: every promoter (enhancer) window of every gene was scanned with FIMO for the biosample's non-redundant motifs (and, for comparison, for the 841 JASPAR motifs), sites were counted per gene and motif, and a ridge regression predicted log2 expression from the counts. Two counting rules: **peak-weighted** (a site is counted once for every window that covers it, so a site under two overlapping peak windows counts twice) and **unweighted** (every genomic site once). The panel reports the held-out Spearman correlation for promoter+enhancer, promoter-only and enhancer-only counts across the 116 biosamples (median ≈ 0.66-0.69, promoter-driven; enhancer-only ≈ 0.41).

The FIMO scans are not rerun here (hours per biosample); `data/k562_demo/count_matrices/` ships the K562 count matrices and this section fits the models on them. The protocol is that of `linear_models/cell_type_specific_predictions_new.py`: 90/10 gene split (`random_state = 42`), standardized counts, `RidgeCV` over α = 10⁻⁶ … 10⁶, Spearman on the held-out genes. For the JASPAR matrices the script drew a random subset of JASPAR motifs of the same size as the de novo set, so those numbers vary slightly with the random draw.

In [ ]:
print(open(f"{DEMO}/count_matrices/README.txt").read())
Z = np.load(f"{DEMO}/count_matrices/{BSSID}_count_matrices.npz"); M = {k: Z[k].astype(np.float64) for k in Z.files}
genes_c, y_c = read_expression(f"{DEMO}/{BSSID}.expression.tsv")
for k in M: print(f"  {k:22s} {M[k].shape}  sites per gene: mean {M[k].sum(1).mean():.0f}")
rs = np.random.RandomState(SEED)
for t in ("pro", "enh"):                     # JASPAR: a random subset of the same size as the de novo set (as in the paper's script)
    n = M[f"{t}_cnncr_weighted"].shape[1]; cols = rs.choice(M[f"{t}_jaspar_weighted"].shape[1], n, replace=False)
    for wgt in ("weighted", "unweighted"): M[f"{t}_jaspar_{wgt}"] = M[f"{t}_jaspar_{wgt}"][:, cols]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
def ridge_model(counts, y):
    """the paper's linear model: 90/10 split, standardized counts, RidgeCV; returns held-out Spearman, the held-out targets and predictions"""
    Xtr, Xte, ytr, yte = train_test_split(counts, y, test_size=0.1, random_state=42)
    sc = StandardScaler().fit(Xtr); m = RidgeCV(alphas=np.logspace(-6, 6, 13)).fit(sc.transform(Xtr), ytr)
    pred = m.predict(sc.transform(Xte)); return spearman(yte, pred), yte, pred, m.alpha_
published = {("cnncr", "weighted", "pro+enh"): 0.763, ("cnncr", "weighted", "pro"): 0.755, ("cnncr", "weighted", "enh"): 0.420,       # K562 values behind Fig. 1h
             ("cnncr", "unweighted", "pro+enh"): 0.743, ("cnncr", "unweighted", "pro"): 0.735, ("cnncr", "unweighted", "enh"): 0.422,
             ("jaspar", "weighted", "pro+enh"): 0.748, ("jaspar", "weighted", "pro"): 0.741, ("jaspar", "weighted", "enh"): 0.425,
             ("jaspar", "unweighted", "pro+enh"): 0.730, ("jaspar", "unweighted", "pro"): 0.725, ("jaspar", "unweighted", "enh"): 0.429}
rows, keep = [], {}
for motifs_ in ("cnncr", "jaspar"):
    for wgt in ("weighted", "unweighted"):
        feats = {"pro": M[f"pro_{motifs_}_{wgt}"], "enh": M[f"enh_{motifs_}_{wgt}"]}; feats["pro+enh"] = np.concatenate([feats["pro"], feats["enh"]], axis=1)
        for fs, Xc in feats.items():
            rho, yte, pred, alpha = ridge_model(Xc, y_c); keep[(motifs_, wgt, fs)] = (yte, pred)
            rows.append(dict(motifs="CNN-CR de novo" if motifs_ == "cnncr" else "JASPAR", counting=wgt, features=fs, n_motifs=Xc.shape[1], alpha=alpha, spearman_heldout=round(rho, 3), published_K562=published[(motifs_, wgt, fs)]))
res = pd.DataFrame(rows); print(res.to_string(index=False))

In [ ]:
from scipy.stats import gaussian_kde
yte, pred = keep[("cnncr", "weighted", "pro+enh")]; yt, yp = yte / np.log2(10), pred / np.log2(10)
z = gaussian_kde(np.vstack([yt, yp]))(np.vstack([yt, yp])); i = z.argsort()
fig, ax = plt.subplots(figsize=(3, 3)); ax.scatter(yt[i], yp[i], c=z[i], s=3, cmap="cividis", rasterized=True)
ax.plot([0, 3], [0, 3], ls="--", color="#66737c", lw=0.8); ax.set_xlim(-0.05, 3.1); ax.set_ylim(-0.05, 3.1); ax.set_box_aspect(1)
ax.set_xlabel("measured expression (log10 FPKM)"); ax.set_ylabel("predicted (log10 FPKM)"); ax.set_title(f"K562, ridge on de novo motif counts\nheld-out Spearman ρ = {spearman(yte, pred):.3f}", fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); fig.savefig(f"{WORK}/fig1h_{BSSID}_linear_scatter.png", dpi=200, transparent=True); plt.show()

Promoter counts carry nearly all of the signal (the enhancer-only model collapses to ρ ≈ 0.42), peak-weighted counting beats unweighted counting, and the biosample's own de novo vocabulary edges out the same number of JASPAR motifs; Fig. 1h is this table for every biosample. Section 4 of the paper (Fig. 2) then asks how much of this signal survives when the ~700-motif vocabulary is replaced by the 43 core motifs shared across biosamples (notebooks 04 and 05).

---
**What the three Figure 1 notebooks wrote** (`$ENHANCER_ROOT/notebook_runs/fig1/BSS00762/`): the trained model(s) and their held-out predictions (01), the extracted per-model motifs and the de-replication folder with the regenerated K562 motif set (02), and the panels `fig1d_*.png`, `fig1e.png`, `fig1f.png`, `fig1h_*.png`.